# A Practical Guide to Measuring Text Similarity

### A hands-on look at how text can be compared by words, structure, and meaning, and how these NLP techniques can be used to evaluate variation in AI-generated answers.

**Joe Domaleski**  
Marketing Data Science

---

## Introduction

Two pieces of text can look different while saying essentially the same thing. They can also look almost identical while making very different claims.

That creates an important measurement problem. If we want to compare customer reviews, survey responses, social media posts, documents, or answers generated by an AI system, what does it actually mean for two texts to be "similar"?

There is no single best similarity score. Different methods measure different things.

In this notebook, we will compare several approaches:

1. Exact string matching
2. Levenshtein similarity
3. Jaccard similarity
4. TF-IDF with cosine similarity
5. Sentence embeddings with cosine similarity

We will start with controlled examples where we know what changed. Then we will use the same methods to compare multiple AI-generated answers to the same marketing question.

The goal is not to declare one method the winner. The goal is to understand what each method sees, what it misses, and which method makes sense for the question we are trying to answer.

## 1. Setup

This notebook is designed to run in **Google Colab**.

Most of the libraries we need are already available in Colab. We will install:

- `rapidfuzz` for normalized Levenshtein similarity
- `sentence-transformers==6.0.1` for sentence embeddings and the optional NLI example

The Sentence Transformers version is pinned so the environment is more reproducible. The embedding model used throughout the notebook is:

`sentence-transformers/all-MiniLM-L6-v2`

The first time the embedding or NLI models run, Colab will download pretrained model files from Hugging Face.

This version also pins the specific Hugging Face model revisions used for the embedding and NLI examples so the published example scores are more reproducible.

In [ ]:
!pip -q install rapidfuzz sentence-transformers==6.0.1

### Import the libraries

We will use:

- `pandas` and `numpy` for data handling
- `matplotlib` for visualization
- `scikit-learn` for TF-IDF and cosine similarity
- `rapidfuzz` for Levenshtein similarity
- `sentence-transformers` for semantic embeddings

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import combinations
from rapidfuzz.distance import Levenshtein
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import mannwhitneyu
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_colwidth", 120)

print("Libraries loaded successfully.")

## 2. Controlled Text Examples

Before comparing a collection of AI answers, it helps to start with examples where we deliberately control the differences.

The examples below are designed to test several situations:

- identical text
- similar meaning with different wording
- high word overlap with opposite meaning
- similar concepts expressed differently
- nearly identical wording with a materially different number
- unrelated text

These examples will help us see why a high similarity score does not always mean two texts agree.

In [ ]:
text_pairs = [
    {
        "pair": "A",
        "description": "Exact match",
        "text_1": "Marketing attribution identifies which channels contribute to a conversion.",
        "text_2": "Marketing attribution identifies which channels contribute to a conversion."
    },
    {
        "pair": "B",
        "description": "Same idea, different wording",
        "text_1": "Marketing attribution identifies which channels contribute to a conversion.",
        "text_2": "Marketing attribution helps determine which marketing channels influenced a customer conversion."
    },
    {
        "pair": "C",
        "description": "Nearly identical wording, opposite direction",
        "text_1": "The campaign increased conversions by 15 percent.",
        "text_2": "The campaign decreased conversions by 15 percent."
    },
    {
        "pair": "D",
        "description": "Related concept, different wording",
        "text_1": "Customer retention measures how well a business keeps existing customers.",
        "text_2": "A strong retention rate means customers continue doing business with the company."
    },
    {
        "pair": "E",
        "description": "Same wording, materially different number",
        "text_1": "Revenue increased by 10 percent last quarter.",
        "text_2": "Revenue increased by 100 percent last quarter."
    },
    {
        "pair": "F",
        "description": "Unrelated topics",
        "text_1": "Email open rates can help marketers evaluate subject line performance.",
        "text_2": "Linear regression estimates the relationship between variables."
    }
]

pairs_df = pd.DataFrame(text_pairs)
pairs_df[["pair", "description", "text_1", "text_2"]]

## 3. Exact Match

The simplest possible comparison is an exact match.

Two texts receive a score of `1` only if every character is identical. Otherwise, they receive `0`.

This is useful when exact duplication matters, such as identifying repeated records or verifying that a generated response is exactly the same as a previous one.

It is not useful for measuring meaning. A single punctuation change makes two texts different even if a human reader would consider them equivalent.

In [ ]:
def exact_match(text_1, text_2):
    return int(text_1 == text_2)

pairs_df["exact_match"] = pairs_df.apply(
    lambda row: exact_match(row["text_1"], row["text_2"]),
    axis=1
)

pairs_df[["pair", "description", "exact_match"]]

## 4. Levenshtein Similarity

Levenshtein distance measures how many single-character edits are needed to turn one string into another.

Those edits can be:

- insertions
- deletions
- substitutions

A raw edit distance can be difficult to compare across texts of different lengths, so we will use a **normalized Levenshtein similarity score** from 0 to 1.

- `1.00` means the strings are identical
- scores closer to `0.00` mean more character-level editing would be required

This method is good at measuring structural or spelling similarity, but it does not understand meaning.

In [ ]:
def levenshtein_similarity(text_1, text_2):
    return Levenshtein.normalized_similarity(text_1, text_2)

pairs_df["levenshtein"] = pairs_df.apply(
    lambda row: levenshtein_similarity(row["text_1"], row["text_2"]),
    axis=1
)

pairs_df[["pair", "description", "levenshtein"]].round(3)

## 5. Jaccard Similarity

Jaccard similarity compares the overlap between two sets.

For text, we can turn each sentence into a set of normalized words and ask:

> What proportion of the unique words found in either text appear in both?

The formula is:

$$
J(A,B) = \frac{|A \cap B|}{|A \cup B|}
$$

A score of `1` means both texts contain the same set of words. A score of `0` means they share no words.

Jaccard similarity ignores word order and frequency. That makes it easy to understand, but it also means important words can be treated the same as unimportant ones.

In [ ]:
def tokenize_words(text):
    return set(re.findall(r"\b\w+\b", text.lower()))

def jaccard_similarity(text_1, text_2):
    words_1 = tokenize_words(text_1)
    words_2 = tokenize_words(text_2)

    union = words_1 | words_2
    if not union:
        return 1.0

    intersection = words_1 & words_2
    return len(intersection) / len(union)

pairs_df["jaccard"] = pairs_df.apply(
    lambda row: jaccard_similarity(row["text_1"], row["text_2"]),
    axis=1
)

pairs_df[["pair", "description", "jaccard"]].round(3)

## 6. TF-IDF with Cosine Similarity

Jaccard similarity treats every word equally. TF-IDF improves on that by giving more weight to terms that are informative across a **corpus** of documents.

TF-IDF stands for:

- **Term Frequency**: how often a term appears in a document
- **Inverse Document Frequency**: how uncommon that term is across the corpus

A key detail is that IDF only makes sense relative to a collection of documents. Instead of fitting a new TF-IDF model separately for every text pair, we will fit **one vectorizer across all twelve controlled texts** and then compare the corresponding rows.

Once each text is represented as a TF-IDF vector, we use **cosine similarity** to compare the vectors.

Because TF-IDF values are nonnegative, cosine similarity in this section falls between 0 and 1:

- values near `1` indicate similar weighted term usage
- values near `0` indicate little overlap in the weighted vocabulary

This is a more meaningful use of TF-IDF than fitting it independently on each pair because words that appear broadly across the controlled corpus receive less weight than more distinctive words.

**Important:** TF-IDF similarity is corpus-dependent. The same two texts can receive different TF-IDF cosine scores when the surrounding corpus changes because IDF weights are recalculated from that corpus. If you run this notebook on your own data, you should expect your TF-IDF scores to differ from the examples here.

In [ ]:
# Build one corpus containing both texts from every controlled pair.
controlled_corpus = []

for item in text_pairs:
    controlled_corpus.extend([item["text_1"], item["text_2"]])

# Fit TF-IDF once across the full controlled corpus.
controlled_tfidf_vectorizer = TfidfVectorizer()
controlled_tfidf_matrix = controlled_tfidf_vectorizer.fit_transform(controlled_corpus)

# Compare the two rows associated with each controlled pair.
controlled_tfidf_scores = []

for pair_index, item in enumerate(text_pairs):
    row_1 = pair_index * 2
    row_2 = row_1 + 1

    score = cosine_similarity(
        controlled_tfidf_matrix[row_1:row_1 + 1],
        controlled_tfidf_matrix[row_2:row_2 + 1]
    )[0, 0]

    controlled_tfidf_scores.append(score)

pairs_df["tfidf_cosine"] = controlled_tfidf_scores

pairs_df[["pair", "description", "tfidf_cosine"]].round(3)


## 7. Sentence Embeddings with Cosine Similarity

Modern NLP gives us another way to compare text: **embeddings**.

An embedding converts text into a numeric vector designed to capture patterns associated with semantic meaning. Sentences that express similar ideas often end up closer together in the embedding space, even when they use different words.

We will use the pretrained `sentence-transformers/all-MiniLM-L6-v2` model.

After creating an embedding for each sentence, we will again use cosine similarity to compare the vectors.

Unlike TF-IDF vectors, general embedding vectors can produce cosine similarities from **-1 to 1**. In practice, this particular model often produces positive values for ordinary English sentence comparisons, but the theoretical range is still -1 to 1.

This gives us a model-based estimate of **semantic similarity**. It is useful for recognizing paraphrases and related ideas, but it is not proof that two statements fully agree or are both correct.

### Practical length limit

`all-MiniLM-L6-v2` uses a maximum sequence length of about **256 word pieces/tokens**. Longer inputs are truncated by the model. That is not a problem for the short examples in this notebook, but it matters when comparing long AI-generated responses. Longer responses may need to be chunked, summarized, or embedded with a model designed for longer contexts.

In [ ]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model_revision = "ea78891063587eb050ed4166b20062eaf978037c"

model = SentenceTransformer(
    embedding_model_name,
    revision=embedding_model_revision
)

# Encode the controlled corpus once.
controlled_embeddings = model.encode(
    controlled_corpus,
    normalize_embeddings=True
)

controlled_embedding_scores = []

for pair_index, item in enumerate(text_pairs):
    row_1 = pair_index * 2
    row_2 = row_1 + 1

    score = float(
        np.dot(
            controlled_embeddings[row_1],
            controlled_embeddings[row_2]
        )
    )

    controlled_embedding_scores.append(score)

pairs_df["embedding_cosine"] = controlled_embedding_scores

pairs_df[["pair", "description", "embedding_cosine"]].round(3)


## 8. Compare the Methods Side by Side

Now we can put all of the scores in one table.

This is where the differences between methods become easier to see.

Pay particular attention to **Pair C**:

> The campaign increased conversions by 15 percent.  
> The campaign decreased conversions by 15 percent.

Almost every word is identical, so several similarity measures may score the pair highly. But the business meaning changes dramatically because one word reverses the conclusion.

Pair E introduces a related problem with numbers:

> Revenue increased by 10 percent last quarter.  
> Revenue increased by 100 percent last quarter.

The wording is nearly identical, yet the magnitude of the business result is dramatically different.

These examples highlight an important limitation to remember throughout this notebook:

> **Similarity is not the same thing as factual agreement, logical agreement, numerical equivalence, or correctness.**

In [ ]:
comparison_columns = [
    "pair",
    "description",
    "exact_match",
    "levenshtein",
    "jaccard",
    "tfidf_cosine",
    "embedding_cosine"
]

comparison_df = pairs_df[comparison_columns].copy()

numeric_cols = [
    "levenshtein",
    "jaccard",
    "tfidf_cosine",
    "embedding_cosine"
]

comparison_df[numeric_cols] = comparison_df[numeric_cols].round(3)
comparison_df

### Visual comparison of the similarity scores

A chart makes it easier to see how the methods react differently to the same text pairs.

The exact-match score is omitted from this chart because it is binary. We will compare the four graded similarity measures.

In [ ]:
plot_df = comparison_df.set_index("pair")[
    ["levenshtein", "jaccard", "tfidf_cosine", "embedding_cosine"]
]

ax = plot_df.plot(
    kind="bar",
    figsize=(11, 6)
)

ax.set_title("Text Similarity Scores by Method")
ax.set_xlabel("Text Pair")
ax.set_ylabel("Similarity Score")
ax.set_ylim(0, 1.05)
ax.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

### A note about preprocessing

The methods in this notebook do not all see exactly the same representation of the text:

- **Levenshtein** compares the raw strings, including case and punctuation.
- **Jaccard** lowercases the text and extracts word tokens with a regular expression.
- **TF-IDF** uses scikit-learn's default tokenization and weighting rules.
- **Embeddings** receive the original text and apply the model's own tokenizer.

That is realistic for how these methods are commonly used, but it also means the differences between scores reflect both the similarity method and its preprocessing choices. The scores should therefore be interpreted as method-specific measurements, not as perfectly controlled apples-to-apples quantities.


## 9. Moving from Two Texts to AI-Generated Answers

Comparing two sentences is useful for learning the mechanics, but the more interesting use case is comparing many answers generated from the same prompt.

Suppose we ask an AI system:

> **What is marketing attribution, and why does it matter?**

Below are eight example AI-generated answers created for this notebook. They intentionally vary in wording, length, detail, and emphasis.

Most communicate broadly similar ideas. One answer also makes a questionable overstatement so we can see whether a similarity score alone is enough to identify a substantive problem.

In a future experiment, these responses could come directly from repeated API calls rather than being stored in the notebook.

In [ ]:
prompt = "What is marketing attribution, and why does it matter?"

ai_answers = {
    "Answer 1": (
        "Marketing attribution is the process of identifying which marketing touchpoints "
        "contributed to a conversion. It matters because it helps marketers understand "
        "which channels and campaigns are influencing results so they can make better "
        "budget and strategy decisions."
    ),
    "Answer 2": (
        "Marketing attribution helps determine which ads, channels, and customer interactions "
        "played a role in producing a conversion. By connecting marketing activity to outcomes, "
        "businesses can evaluate performance and allocate resources more effectively."
    ),
    "Answer 3": (
        "Attribution is a way to assign credit for a sale or conversion across the marketing "
        "touchpoints a customer encountered. It is useful because marketers can see which "
        "parts of the customer journey appear to contribute most to business results."
    ),
    "Answer 4": (
        "Marketing attribution connects customer conversions back to the marketing interactions "
        "that preceded them. The goal is to understand what influenced the outcome and use that "
        "information to improve campaigns, channel mix, and spending decisions."
    ),
    "Answer 5": (
        "Marketing attribution measures the relationship between marketing touchpoints and "
        "conversions. It matters because marketers rarely have unlimited budgets, so they need "
        "evidence about which activities deserve more investment and which may deserve less."
    ),
    "Answer 6": (
        "Attribution tries to answer a practical question: which marketing efforts helped produce "
        "this result? Different attribution models divide credit differently, but all are attempts "
        "to connect marketing exposure with outcomes and improve decision-making."
    ),
    "Answer 7": (
        "Marketing attribution is the practice of tracing conversions to the channels or interactions "
        "that influenced them. It can help teams compare marketing performance, understand customer "
        "journeys, and make more informed decisions about future campaigns."
    ),
    "Answer 8": (
        "Marketing attribution identifies the single marketing channel that caused a conversion. "
        "It matters because the last interaction before the sale should receive all of the credit, "
        "making last-click attribution the most accurate method for every business."
    )
}

answers_df = pd.DataFrame(
    [{"answer": name, "text": text} for name, text in ai_answers.items()]
)

print("Prompt:")
print(prompt)
print("\nAI-generated answers:")
answers_df

# A second, clearly different marketing question gives us a better baseline
# for interpreting semantic similarity scores.
baseline_prompt = "What is customer lifetime value, and why does it matter?"

baseline_answers = {
    "Baseline 1": (
        "Customer lifetime value estimates the total economic value a customer is expected "
        "to generate over the course of the relationship. It helps businesses decide how much "
        "they can reasonably spend to acquire and retain customers."
    ),
    "Baseline 2": (
        "CLV is an estimate of the revenue or profit associated with a customer across the full "
        "relationship, not just one transaction. It matters because it connects acquisition "
        "costs, retention, and long-term profitability."
    ),
    "Baseline 3": (
        "Customer lifetime value helps companies understand the long-term value of different "
        "customers or segments. That can guide decisions about marketing investment, service, "
        "retention programs, and acquisition spending."
    ),
    "Baseline 4": (
        "Lifetime value looks beyond a single purchase and estimates what a customer may be "
        "worth over time. Marketers use it to evaluate whether acquisition and retention costs "
        "make financial sense."
    ),
    "Baseline 5": (
        "Customer lifetime value is a forward-looking estimate of the value created by a customer "
        "relationship. It can help prioritize high-value segments and balance short-term campaign "
        "performance with longer-term profitability."
    ),
    "Baseline 6": (
        "CLV answers a practical question: how valuable is this customer relationship over its "
        "expected life? The metric is useful for budgeting, segmentation, retention planning, "
        "and comparing customer acquisition cost with expected value."
    ),
    "Baseline 7": (
        "Customer lifetime value estimates how much value a business expects to receive from a "
        "customer over time. It matters because customers with similar first purchases can have "
        "very different long-term value."
    ),
    "Baseline 8": (
        "Lifetime value combines assumptions about customer spending, purchase frequency, margin, "
        "and retention to estimate long-term economic value. Businesses can use it to make more "
        "informed marketing and customer strategy decisions."
    )
}

baseline_df = pd.DataFrame(
    [{"answer": name, "text": text} for name, text in baseline_answers.items()]
)

print("\nBaseline prompt:")
print(baseline_prompt)
print("\nBaseline answers:")
baseline_df


## 10. Compare Each AI Answer to a Reference Answer

One common evaluation strategy is to compare every response with a reference.

For demonstration purposes, we will use **Answer 1** as the reference. This does not mean it is the objectively correct or ideal answer. It simply gives us a consistent baseline.

Before comparing the responses, we will fit TF-IDF **once across all eight AI answers** and create all eight sentence embeddings **once**. We can then reuse those representations throughout the rest of the notebook.

We will calculate:

- Levenshtein similarity
- Jaccard similarity
- TF-IDF cosine similarity
- embedding cosine similarity

This lets us see whether an answer can be lexically different but semantically similar, or lexically similar while making a different substantive claim.

In [ ]:
answer_names = list(ai_answers.keys())
answer_texts = list(ai_answers.values())

# Fit TF-IDF once across all eight AI answers.
ai_tfidf_vectorizer = TfidfVectorizer()
ai_tfidf_matrix = ai_tfidf_vectorizer.fit_transform(answer_texts)

# Encode all eight AI answers once.
answer_embeddings = model.encode(
    answer_texts,
    normalize_embeddings=True
)

# Precompute the semantic similarity matrix once and reuse it later.
embedding_matrix = cosine_similarity(answer_embeddings)

reference_name = "Answer 1"
reference_index = answer_names.index(reference_name)
reference_text = ai_answers[reference_name]

reference_results = []

for i, (answer_name, answer_text) in enumerate(ai_answers.items()):
    reference_results.append({
        "answer": answer_name,
        "levenshtein": levenshtein_similarity(reference_text, answer_text),
        "jaccard": jaccard_similarity(reference_text, answer_text),
        "tfidf_cosine": cosine_similarity(
            ai_tfidf_matrix[reference_index:reference_index + 1],
            ai_tfidf_matrix[i:i + 1]
        )[0, 0],
        "embedding_cosine": embedding_matrix[reference_index, i]
    })

reference_df = pd.DataFrame(reference_results).round(3)
reference_df


### A note about the problematic answer

Answer 8 contains many of the same marketing-attribution terms as the other answers, but it makes a much stronger claim:

> "...the last interaction before the sale should receive all of the credit..."

A text-similarity model may still consider that answer similar because it is about the same topic and uses much of the same vocabulary.

This demonstrates an important distinction:

- **Text similarity** asks whether two texts resemble each other.
- **Factual evaluation** asks whether the claims are supported.
- **Logical evaluation** asks whether the reasoning is sound.
- **Quality evaluation** asks whether the answer is useful for the intended purpose.

Those are related questions, but they are not interchangeable.

## 11. Pairwise Semantic Similarity Across All AI Answers

When we have many responses, comparing everything to a single reference can hide useful information.

Instead, we can compare every answer with every other answer.

For eight answers, that produces an 8 × 8 similarity matrix. The diagonal will always equal 1 because every answer is identical to itself.

Here we will use sentence embeddings because our main interest is semantic similarity.

In [ ]:
embedding_similarity_df = pd.DataFrame(
    embedding_matrix,
    index=answer_names,
    columns=answer_names
).round(3)

embedding_similarity_df


## 12. Visualize the Similarity Matrix

A heatmap makes the pairwise comparison easier to scan.

Values closer to 1 indicate greater semantic similarity according to the embedding model.

The heatmap is useful when an experiment contains enough responses that reading every pair manually becomes impractical.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

image = ax.imshow(
    embedding_similarity_df.values,
    vmin=0,
    vmax=1
)

ax.set_xticks(range(len(answer_names)))
ax.set_yticks(range(len(answer_names)))
ax.set_xticklabels(answer_names, rotation=45, ha="right")
ax.set_yticklabels(answer_names)

for i in range(len(answer_names)):
    for j in range(len(answer_names)):
        ax.text(
            j,
            i,
            f"{embedding_similarity_df.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

ax.set_title("Pairwise Semantic Similarity of AI-Generated Answers")
fig.colorbar(image, ax=ax, label="Cosine Similarity")

plt.tight_layout()
plt.show()

## 13. Find the Most and Least Similar Answer Pairs

A matrix is useful visually, but we may also want direct summary statistics.

We will calculate every unique answer pair and identify:

- the most semantically similar pair
- the least semantically similar pair

With eight answers there are only 28 unique pairs. With hundreds of answers there can be tens of thousands of pairwise comparisons, so automating this step becomes increasingly useful.

In [ ]:
pairwise_results = []

for i, j in combinations(range(len(answer_names)), 2):
    pairwise_results.append({
        "answer_1": answer_names[i],
        "answer_2": answer_names[j],
        "embedding_similarity": embedding_matrix[i, j]
    })

pairwise_df = pd.DataFrame(pairwise_results).sort_values(
    "embedding_similarity",
    ascending=False
).reset_index(drop=True)

pairwise_df["embedding_similarity"] = pairwise_df["embedding_similarity"].round(3)

print("Most similar pair:")
display(pairwise_df.head(1))

print("\nLeast similar pair:")
display(pairwise_df.tail(1))

print("\nAll unique pairs:")
pairwise_df

## 14. Which Answer Is Most Representative?

We can also ask which response is most similar, on average, to all of the other responses.

One simple approach is to calculate the mean pairwise semantic similarity for each answer, excluding its similarity with itself.

The response with the highest average similarity can be thought of as the most **central** or **representative** answer in this particular set.

This does **not** mean it is the best or most accurate answer. It only means it is closest to the center of what the group is saying.

In [ ]:
average_similarities = []

for i, answer_name in enumerate(answer_names):
    other_scores = np.delete(embedding_matrix[i], i)

    average_similarities.append({
        "answer": answer_name,
        "average_similarity_to_others": other_scores.mean()
    })

centrality_df = pd.DataFrame(average_similarities).sort_values(
    "average_similarity_to_others",
    ascending=False
).reset_index(drop=True)

centrality_df["average_similarity_to_others"] = (
    centrality_df["average_similarity_to_others"].round(3)
)

centrality_df

### A useful warning: unusual does not mean wrong

The centrality ranking contains a result worth noticing.

**Answer 8**, which deliberately contains an overly strong last-click attribution claim, is not the biggest semantic outlier. **Answer 6**, which is substantively reasonable, has the lowest average similarity to the rest of the answers.

That gives us a sharper version of an important warning:

> A semantic outlier detector can identify answers that are unusual, but unusual is not the same thing as incorrect.

In this small example, a reasonable answer can look more unusual than a problematic answer that uses familiar attribution language.

We should be careful not to over-explain why from this small sample alone. Answer 6 differs in wording and framing, but the notebook does not establish which specific feature caused its lower centrality.


## 15. Compare Multiple Similarity Methods Across the AI Answers

Semantic embeddings are powerful, but it is useful to see how different methods behave on the same collection.

The next table summarizes the similarity across every unique pair of AI answers for four methods.

For TF-IDF, we reuse the matrix fit once across all eight answers. For embeddings, we reuse the eight embeddings and the semantic similarity matrix already computed. No response is re-encoded for every pair.

This is both more efficient and more representative of how these methods should be used at scale.

In [ ]:
method_pair_scores = []

for i, j in combinations(range(len(answer_names)), 2):
    text_1 = answer_texts[i]
    text_2 = answer_texts[j]

    method_pair_scores.append({
        "answer_1": answer_names[i],
        "answer_2": answer_names[j],
        "levenshtein": levenshtein_similarity(text_1, text_2),
        "jaccard": jaccard_similarity(text_1, text_2),
        "tfidf_cosine": cosine_similarity(
            ai_tfidf_matrix[i:i + 1],
            ai_tfidf_matrix[j:j + 1]
        )[0, 0],
        "embedding_cosine": embedding_matrix[i, j]
    })

all_methods_df = pd.DataFrame(method_pair_scores)

method_summary_df = (
    all_methods_df[
        ["levenshtein", "jaccard", "tfidf_cosine", "embedding_cosine"]
    ]
    .agg(["mean", "min", "max"])
    .T
    .round(3)
)

method_summary_df


## 16. Add a Cross-Question Baseline

A mean embedding similarity such as `0.84` is not a calibrated probability. By itself, it does not tell us how unusually similar the answers are.

A better comparison is to create a second set of answers to a clearly different marketing question and compare:

1. **Within attribution**: attribution answers compared with other attribution answers
2. **Within CLV**: customer lifetime value answers compared with other CLV answers
3. **Cross-question**: attribution answers compared with CLV answers

This gives us an empirical baseline for this notebook instead of treating one isolated unrelated pair as a universal "floor."

If the same-question distributions sit well above the cross-question distribution, we have stronger evidence that the embedding model is distinguishing shared semantic content rather than simply producing generally high cosine scores.

In [ ]:
baseline_names = list(baseline_answers.keys())
baseline_texts = list(baseline_answers.values())

baseline_embeddings = model.encode(
    baseline_texts,
    normalize_embeddings=True
)

baseline_embedding_matrix = cosine_similarity(baseline_embeddings)

cross_question_matrix = cosine_similarity(
    answer_embeddings,
    baseline_embeddings
)

def upper_triangle_values(matrix):
    """Return unique off-diagonal pairwise similarities from a square matrix."""
    return matrix[np.triu_indices_from(matrix, k=1)]

within_attribution = upper_triangle_values(embedding_matrix)
within_clv = upper_triangle_values(baseline_embedding_matrix)
cross_question = cross_question_matrix.ravel()

# Inspect the lowest-similarity CLV pair before interpreting the distribution.
clv_pair_rows = []

for i, j in combinations(range(len(baseline_names)), 2):
    clv_pair_rows.append({
        "answer_1": baseline_names[i],
        "answer_2": baseline_names[j],
        "embedding_similarity": baseline_embedding_matrix[i, j]
    })

clv_pair_df = (
    pd.DataFrame(clv_pair_rows)
    .sort_values("embedding_similarity")
    .reset_index(drop=True)
)

lowest_clv_pair = clv_pair_df.iloc[0]

print("Lowest-similarity CLV pair:")
display(clv_pair_df.head(1).round(3))

print("\nFirst response:")
print(baseline_answers[lowest_clv_pair["answer_1"]])

print("\nSecond response:")
print(baseline_answers[lowest_clv_pair["answer_2"]])

baseline_summary_df = pd.DataFrame({
    "comparison": [
        "Within attribution answers",
        "Within CLV answers",
        "Cross-question: attribution vs. CLV"
    ],
    "mean": [
        within_attribution.mean(),
        within_clv.mean(),
        cross_question.mean()
    ],
    "min": [
        within_attribution.min(),
        within_clv.min(),
        cross_question.min()
    ],
    "max": [
        within_attribution.max(),
        within_clv.max(),
        cross_question.max()
    ],
    "n_comparisons": [
        len(within_attribution),
        len(within_clv),
        len(cross_question)
    ]
}).round(3)

print("\nEmbedding baseline summary:")
baseline_summary_df


### Inspect the minimum before interpreting it

The minimum within-CLV similarity falls into the same range as some cross-question comparisons, so it deserves inspection before we build a story around it.

The code above prints the lowest-similarity CLV pair and both complete answers. The appropriate next step is to describe what is actually different about those texts without assuming a cause. If both are reasonable responses to the CLV prompt, the result is evidence that same-question answers can still vary substantially. If one is clearly malformed or off-topic, the answer set should be corrected and every downstream result regenerated.

In both question sets, answers to the same prompt have higher **average** semantic similarity than the cross-question comparisons. The size of that separation, however, is question-dependent. Because the CLV range overlaps the cross-question range, there is no universal cosine-similarity threshold that means two answers came from the same question.


### Visualize the baseline distributions

The distributions are more informative than a single benchmark value. This plot lets us compare the spread of same-question similarities with the spread of cross-question similarities.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.boxplot(
    [within_attribution, within_clv, cross_question],
    tick_labels=[
        "Within\nattribution",
        "Within\nCLV",
        "Cross-question"
    ]
)

ax.set_title("Semantic Similarity: Same Question vs. Different Question")
ax.set_ylabel("Embedding Cosine Similarity")
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()


## 17. Compare Same-Question vs. Cross-Question Separation Across Methods

The embedding baseline is useful, but the same comparison should be made for Levenshtein, Jaccard, TF-IDF, and embeddings.

The raw similarity scores are not directly comparable across methods because each method has a different scale and distribution. We therefore calculate two descriptive separation measures.

### Separation margin

**separation margin = minimum within-question score − maximum cross-question score**

A positive margin means the observed score ranges do not overlap. A negative margin means some same-question and cross-question scores overlap.

### Common-language effect size

We also calculate the probability that a randomly selected within-question pair receives a higher similarity score than a randomly selected cross-question pair. This common-language effect size is equivalent to an AUC-style rank statistic in this setting.

- `1.0` means complete separation in this observed sample.
- `0.5` means no useful rank separation.

We calculate it as `U / (n_within × n_cross)` using the Mann-Whitney U statistic.

This is **descriptive only**. The pairwise scores are not independent because the same underlying answers appear in multiple pairs. With only eight answers per question, this is a demonstration rather than a population estimate, so we do not report p-values.

In [ ]:
# Fit one TF-IDF model across all 16 AI answers so every TF-IDF
# comparison uses the same corpus, vocabulary, and IDF weights.
combined_ai_texts = answer_texts + baseline_texts

combined_tfidf_vectorizer = TfidfVectorizer()
combined_tfidf_matrix = combined_tfidf_vectorizer.fit_transform(combined_ai_texts)

n_attr = len(answer_texts)

attr_tfidf = combined_tfidf_matrix[:n_attr]
clv_tfidf = combined_tfidf_matrix[n_attr:]

attr_tfidf_matrix = cosine_similarity(attr_tfidf)
clv_tfidf_matrix = cosine_similarity(clv_tfidf)
cross_tfidf_matrix = cosine_similarity(attr_tfidf, clv_tfidf)


def pairwise_string_scores(texts, method):
    """Return unique within-set pairwise scores for a string-based method."""
    return np.array([
        method(texts[i], texts[j])
        for i, j in combinations(range(len(texts)), 2)
    ])


def cross_string_scores(texts_1, texts_2, method):
    """Return every cross-set pairwise score for a string-based method."""
    return np.array([
        method(text_1, text_2)
        for text_1 in texts_1
        for text_2 in texts_2
    ])


attr_lev = pairwise_string_scores(answer_texts, levenshtein_similarity)
clv_lev = pairwise_string_scores(baseline_texts, levenshtein_similarity)
cross_lev = cross_string_scores(answer_texts, baseline_texts, levenshtein_similarity)

attr_jaccard = pairwise_string_scores(answer_texts, jaccard_similarity)
clv_jaccard = pairwise_string_scores(baseline_texts, jaccard_similarity)
cross_jaccard = cross_string_scores(answer_texts, baseline_texts, jaccard_similarity)

attr_tfidf_scores = upper_triangle_values(attr_tfidf_matrix)
clv_tfidf_scores = upper_triangle_values(clv_tfidf_matrix)
cross_tfidf_scores = cross_tfidf_matrix.ravel()

attr_embed = within_attribution
clv_embed = within_clv
cross_embed = cross_question


method_distributions = {
    "Levenshtein": {
        "attribution": attr_lev,
        "clv": clv_lev,
        "cross": cross_lev
    },
    "Jaccard": {
        "attribution": attr_jaccard,
        "clv": clv_jaccard,
        "cross": cross_jaccard
    },
    "TF-IDF + cosine": {
        "attribution": attr_tfidf_scores,
        "clv": clv_tfidf_scores,
        "cross": cross_tfidf_scores
    },
    "Embeddings + cosine": {
        "attribution": attr_embed,
        "clv": clv_embed,
        "cross": cross_embed
    }
}


def descriptive_discrimination(within_scores, cross_scores):
    """
    Summarize observed same-question vs. cross-question separation.

    The common-language effect size is descriptive only because the
    pairwise observations reuse the same underlying texts.
    """
    u_stat = mannwhitneyu(
        within_scores,
        cross_scores,
        alternative="greater"
    ).statistic

    return {
        "mean_within": np.mean(within_scores),
        "mean_cross": np.mean(cross_scores),
        "within_min": np.min(within_scores),
        "cross_max": np.max(cross_scores),
        "separation_margin": np.min(within_scores) - np.max(cross_scores),
        "common_language_effect": (
            u_stat / (len(within_scores) * len(cross_scores))
        )
    }


discrimination_rows = []

for method_name, distributions in method_distributions.items():
    for question_name, within_key in [
        ("Attribution", "attribution"),
        ("CLV", "clv")
    ]:
        stats = descriptive_discrimination(
            distributions[within_key],
            distributions["cross"]
        )

        discrimination_rows.append({
            "method": method_name,
            "question_set": question_name,
            **stats
        })

discrimination_df = pd.DataFrame(discrimination_rows).round(3)
discrimination_df


### How to read this table

The **separation margin** directly shows whether the observed same-question and cross-question ranges overlap.

The **common-language effect** is scale-free, so it is more appropriate for comparing methods that otherwise produce very different raw score ranges.

An observed value of `1.000` should not be treated as proof of perfect future classification. It means only that every within-question score exceeded every cross-question score in this small demonstration set. If several methods hit a ceiling for attribution, the CLV results become more informative because that question produces more variation.


## 18. The Contradiction Trap

One of the easiest mistakes to make with similarity metrics is assuming that a high score means two statements agree.

Consider these sentences again:

- **The campaign increased conversions by 15 percent.**
- **The campaign decreased conversions by 15 percent.**

They have nearly identical structure and vocabulary. Only one word changes, but that one word reverses the business conclusion.

Let's isolate this example and look at the scores again.

In [ ]:
contradiction_results = (
    pairs_df.loc[
        pairs_df["pair"] == "C",
        ["levenshtein", "jaccard", "tfidf_cosine", "embedding_cosine"]
    ]
    .reset_index(drop=True)
    .round(3)
)

contradiction_results


The lesson is straightforward:

> **Similarity measures are measurements, not judgment.**

A high score can tell us that two texts share structure, vocabulary, or semantic context. It cannot automatically tell us that both are true, that they agree on every important detail, or that they are equally useful.

For AI evaluation, similarity analysis should usually be one part of a broader evaluation process.

### A second trap: numbers can change the business meaning

Text-similarity methods can also miss the importance of a numerical change.

Compare:

- **Revenue increased by 10 percent last quarter.**
- **Revenue increased by 100 percent last quarter.**

Only one character changes, but the second claim describes a result ten times as large.

For marketing and analytics work, that distinction can be more important than the overall textual similarity. A high similarity score should therefore not be interpreted as evidence that two responses are numerically equivalent.

In [ ]:
numeric_difference_results = (
    pairs_df.loc[
        pairs_df["pair"] == "E",
        ["levenshtein", "jaccard", "tfidf_cosine", "embedding_cosine"]
    ]
    .reset_index(drop=True)
    .round(3)
)

numeric_difference_results


## 19. Natural Language Inference: A Different Question

The contradiction example exposes a limitation of semantic similarity, but Natural Language Inference, or **NLI**, is designed for a different question.

NLI usually classifies a pair of statements as:

- **entailment**: one statement supports or implies the other
- **contradiction**: the statements conflict
- **neutral**: neither entailment nor contradiction is established

A cross-encoder processes both texts together instead of embedding each independently.

We will test three cases:

1. increased vs. decreased conversions
2. 10 percent vs. 100 percent revenue growth
3. Answer 1 vs. the deliberately problematic Answer 8

The last test matters because NLI evaluates the relationship **between two claims**. It is not a general fact checker. A statement can be wrong about the world without directly contradicting another statement.

In [ ]:
nli_model_name = "cross-encoder/nli-deberta-v3-xsmall"
nli_model_revision = "4656f516432a9e42a107681432339720e152807f"

nli_model = CrossEncoder(
    nli_model_name,
    revision=nli_model_revision
)

nli_pairs = [
    (
        "The campaign increased conversions by 15 percent.",
        "The campaign decreased conversions by 15 percent."
    ),
    (
        "Revenue increased by 10 percent last quarter.",
        "Revenue increased by 100 percent last quarter."
    ),
    (
        ai_answers["Answer 1"],
        ai_answers["Answer 8"]
    )
]

nli_pair_names = [
    "Increased vs. decreased",
    "10 percent vs. 100 percent",
    "Answer 1 vs. Answer 8"
]

nli_scores = nli_model.predict(
    nli_pairs,
    apply_softmax=True
)

raw_id2label = nli_model.model.config.id2label

id2label = {
    int(label_id): str(label_name).lower()
    for label_id, label_name in raw_id2label.items()
}

nli_results = []

for pair_name, scores in zip(nli_pair_names, nli_scores):
    predicted_index = int(np.argmax(scores))
    predicted_label = id2label[predicted_index]

    row = {
        "comparison": pair_name,
        "predicted_label": predicted_label
    }

    for class_index, class_probability in enumerate(scores):
        row[id2label[class_index]] = class_probability

    nli_results.append(row)

nli_results_df = pd.DataFrame(nli_results).round(3)
nli_results_df


## 20. Scaling the Approach to Future AI API Experiments

The examples above use small response collections so the notebook stays readable.

The same workflow can be extended to dozens, hundreds, or thousands of responses:

1. Send the same prompt to an AI model many times.
2. Store each response with metadata such as model, timestamp, temperature, and run number.
3. Encode every response **once**.
4. Compute pairwise similarity from the resulting embedding matrix.
5. Summarize the distribution rather than trying to read every pair.
6. Identify clusters, unusual responses, and representative responses.
7. Compare similarity across models, prompts, settings, or time periods.

For `n` responses, there are:

$$
\frac{n(n-1)}{2}
$$

unique response pairs.

For 100 responses, that is 4,950 pairs. For 1,000 responses, it is 499,500.

The raw cosine calculations are not necessarily the hardest part. Once 1,000 normalized 384-dimensional embeddings exist, pairwise cosine similarity can be computed efficiently with matrix operations.

The larger challenge is the **n-squared relationship structure** and, more importantly, human interpretation. Nobody wants to manually inspect 499,500 response pairs.

At scale, the goal is therefore to summarize:

- the similarity distribution
- clusters
- outliers
- representative responses
- required facts
- numerical consistency
- source or citation consistency

That turns text similarity into a screening and summarization tool rather than a substitute for human or factual evaluation.

In [ ]:
def analyze_response_collection(responses, model):
    """
    Encode each response once and return a pairwise cosine-similarity matrix.

    Parameters
    ----------
    responses : dict
        Mapping of response names to response text.
    model : SentenceTransformer
        Loaded sentence embedding model.

    Returns
    -------
    pandas.DataFrame
        Pairwise cosine-similarity matrix.
    """
    names = list(responses.keys())
    texts = list(responses.values())

    embeddings = model.encode(
        texts,
        normalize_embeddings=True
    )

    matrix = embeddings @ embeddings.T

    return pd.DataFrame(
        matrix,
        index=names,
        columns=names
    )


scaled_example = analyze_response_collection(ai_answers, model)
scaled_example.round(3)


## 21. Optional Template for Your Own AI Responses

To analyze your own results later, replace the example dictionary below with responses collected from an API, CSV file, spreadsheet, or another source.

The rest of the analysis can stay essentially the same.

In [ ]:
my_responses = {
    "Run 1": "Paste or load the first AI response here.",
    "Run 2": "Paste or load the second AI response here.",
    "Run 3": "Paste or load the third AI response here."
}

# Uncomment after replacing the placeholder text:
# my_similarity_matrix = analyze_response_collection(my_responses, model)
# display(my_similarity_matrix.round(3))

## 22. Key Takeaways

There is no universal text-similarity score.

Each method answers a somewhat different question:

| Method | What It Mostly Measures | Useful For | Important Limitation |
|---|---|---|---|
| Exact match | Character-for-character equality | Duplicate detection, reproducibility | Any change makes the texts different |
| Levenshtein | Character-level editing distance | Typos, revisions, near-duplicate strings | Does not understand meaning |
| Jaccard | Shared unique words | Simple vocabulary overlap | Ignores order and importance |
| TF-IDF + cosine | Corpus-weighted term overlap | Document similarity, search, classical NLP | Still tied closely to vocabulary |
| Embeddings + cosine | Semantic proximity | Paraphrases, AI responses, meaning-level comparison | Similarity does not guarantee agreement or correctness |
| NLI cross-encoder | Entailment, contradiction, or neutrality | Checking relationships between claims | Slower because text pairs must be evaluated together |

For comparing AI-generated answers, embeddings are especially useful because different responses can express similar ideas using very different words.

But several cautions matter:

- Similarity is not factual accuracy.
- Similarity is not numerical equivalence.
- A semantic outlier is not necessarily a bad answer.
- An incorrect answer can still sound semantically typical.
- Embedding cosine scores are not calibrated probabilities.
- Long responses may be truncated by the embedding model.
- Cross-method comparisons also reflect different preprocessing choices.

A broader AI-evaluation framework may eventually combine:

- semantic similarity
- factual consistency
- numerical consistency
- required facts
- citation or source agreement
- length and structure
- tone
- outlier detection

That turns text similarity from an abstract NLP concept into a practical tool for evaluating variation across many AI-generated answers.

## 23. Next Steps

This notebook provides the foundation for a larger experiment.

A future project could repeatedly call an AI API with the same prompt and measure how much the answers vary across dozens, hundreds, or thousands of runs.

Possible extensions include:

- comparing different models
- changing temperature or other generation settings
- clustering semantically similar answers
- comparing within-prompt and cross-prompt similarity distributions
- visualizing embeddings with PCA or UMAP
- tracking whether key facts appear consistently
- checking numerical consistency
- applying NLI to likely contradictions
- identifying semantic outliers
- comparing automated metrics with human ratings

The important idea is that AI variation can be measured, but no single score captures every dimension of answer quality.

## 24. Resources

Alammar, Jay, and Maarten Grootendorst. *Hands-On Large Language Models: Language Understanding and Generation*. O’Reilly Media, 2024. https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/

Domaleski, Joe. “A Marketer’s Guide to NLP: How Machines Actually Process and Understand Language.” *Marketing Data Science*, 19 Oct. 2025. https://blog.marketingdatascience.ai/a-marketers-guide-to-nlp-how-machines-actually-process-and-understand-language-3d452febb3de

Domaleski, Joe. “Sentiment Analysis of Online Reviews Using R.” *Marketing Data Science*, 22 Sept. 2024. https://blog.marketingdatascience.ai/sentiment-analysis-of-online-reviews-using-r-e2afbc9fcc68

Domaleski, Joe. “UBCF vs. IBCF: Comparing Marketing Recommendation System Algorithms in R.” *Marketing Data Science*, 6 Apr. 2025. https://blog.marketingdatascience.ai/ubcf-vs-ibcf-comparing-marketing-recommendation-system-algorithms-in-r-38ff36bf05d3

Kocaman, Ahmet Münir. “How to Measure Text Similarity: A Comprehensive Guide.” *Medium*, 7 Oct. 2023. https://medium.com/@ahmetmnirkocaman/how-to-measure-text-similarity-a-comprehensive-guide-6c6f24fc01fe

Ladd, John R. “Understanding and Using Common Similarity Measures for Text Analysis.” *Programming Historian*, no. 9, 5 May 2020. https://doi.org/10.46430/phen0089

Levy, Avivit, B. Riva Shalom, and Michal Chalamish. “A Guide to Similarity Measures.” *arXiv*, 7 Aug. 2024. https://arxiv.org/abs/2408.07706

Reimers, Nils, and Iryna Gurevych. “all-MiniLM-L6-v2.” *Hugging Face*, 2021. https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Vaswani, Ashish, et al. “Attention Is All You Need.” *Advances in Neural Information Processing Systems*, vol. 30, 2017, pp. 5998–6008. https://arxiv.org/abs/1706.03762

Wang, Wenhui, et al. “MiniLM: Deep Self-Attention Distillation for Task-Agnostic Compression of Pre-Trained Transformers.” *arXiv*, 2020. https://arxiv.org/abs/2002.10957